# QM640 · 06 Instruments I5 (small language model) and I6 (fine-tuned with LoRA)
**Runtime → Change runtime type → L4 GPU**, then *Runtime → Run all*. About 2 hours (roughly 4–5 compute units). Best run after notebook 05, so the RQ3 context runs are included; if 05 has not run, those cells are skipped and can be run later.

If the session stops, run all again: finished work is reused.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import random, time, json
DATA_ROOT = Path('/content/drive/MyDrive/Walsh DBA/Doctoral Research/QM640/Data')
!pip -q install -U peft
MODEL_ID = 'Qwen/Qwen3-4B-Instruct-2507'; MODEL_NAME = 'Qwen3-4B-Instruct-2507'   # open 4-billion-parameter model (Apache 2.0 licence), run on the Colab GPU
BATCH = 16                       # prompts per generation batch
LORA_R = 16                      # LoRA rank
N_TRAIN_PER_DECISION = 3000      # fine-tuning examples per decision (four decisions)
PER_CLASS_CAP = 400              # cap per class so rare classes are represented
UNITS_PER_HOUR = 2.0             # approximate Colab compute units per hour on an L4 (check View resources)
USD_PER_UNIT = 0.10              # Colab pay-as-you-go price: about US$10 per 100 units
def load_model():
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    assert torch.cuda.is_available(), 'Select an L4 GPU: Runtime > Change runtime type'
    tok = AutoTokenizer.from_pretrained(MODEL_ID); tok.padding_side = 'left'
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='cuda'); base.eval()
    class LM:
        def __init__(self): self.model, self.adapter = base, False
        def chat(self, p): return tok.apply_chat_template([{'role': 'user', 'content': p}], tokenize=False, add_generation_prompt=True)
        @torch.no_grad()
        def _gen(self, prompts, max_new):
            enc = tok([self.chat(p) for p in prompts], return_tensors='pt', padding=True).to('cuda')
            out = self.model.generate(**enc, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None, top_k=None,
                                      output_scores=True, return_dict_in_generate=True, pad_token_id=tok.eos_token_id)
            gen = out.sequences[:, enc['input_ids'].shape[1]:]
            lp = torch.stack(out.scores, 1).float().log_softmax(-1).gather(-1, gen.unsqueeze(-1)).squeeze(-1)
            keep = torch.cumprod((gen != tok.eos_token_id).int(), 1).bool()       # tokens up to the end of the answer
            conf = torch.exp((lp * keep).sum(1)).tolist()                            # probability of the whole answer
            return tok.batch_decode(gen, skip_special_tokens=True), conf, enc['attention_mask'].sum(1).tolist()
        def generate(self, prompts, max_new):
            if hasattr(self.model, 'disable_adapter') and not self.adapter:
                with self.model.disable_adapter(): return self._gen(prompts, max_new)
            return self._gen(prompts, max_new)
        def use_adapter(self, on): self.adapter = on and hasattr(self.model, 'disable_adapter')
        def train_lora(self, examples, out_dir):
            from peft import LoraConfig, get_peft_model, PeftModel
            if (out_dir/'adapter_config.json').exists():                          # reuse a finished fine-tune
                self.model = PeftModel.from_pretrained(base, str(out_dir)); self.model.eval(); self.adapter = True; print('adapter loaded'); return
            cfg = LoraConfig(r=LORA_R, lora_alpha=2*LORA_R, lora_dropout=0.05, task_type='CAUSAL_LM',
                             target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
            m = get_peft_model(base, cfg); base.config.use_cache = False; m.gradient_checkpointing_enable(); m.enable_input_require_grads()
            for p in m.parameters():
                if p.requires_grad: p.data = p.data.float()
            m.print_trainable_parameters()
            data = []
            for prompt, ans in examples:
                a = tok(self.chat(prompt), add_special_tokens=False)['input_ids']; b = tok(ans + tok.eos_token, add_special_tokens=False)['input_ids']
                data.append(((a + b)[-1024:], ([-100]*len(a) + b)[-1024:]))
            opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=2e-4, weight_decay=0.0)
            bs, accum = 4, 4; steps = len(data)//bs; sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s+1)/30) * max(0.0, 1 - s/(steps/accum)))
            order = sorted(range(len(data)), key=lambda i: len(data[i][0])); batches = [order[i:i+bs] for i in range(0, len(order), bs)]; random.shuffle(batches)
            m.train(); t0 = time.time()
            for s, bidx in enumerate(batches):
                L = max(len(data[i][0]) for i in bidx)
                ids = torch.tensor([[tok.pad_token_id]*(L-len(data[i][0])) + data[i][0] for i in bidx]).cuda()
                lab = torch.tensor([[-100]*(L-len(data[i][1])) + data[i][1] for i in bidx]).cuda()
                att = (torch.arange(L)[None, :] >= torch.tensor([L-len(data[i][0]) for i in bidx])[:, None]).long().cuda()
                with torch.autocast('cuda', dtype=torch.bfloat16): loss = m(input_ids=ids, attention_mask=att, labels=lab).loss / accum
                loss.backward()
                if (s+1) % accum == 0: torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step(); sched.step(); opt.zero_grad()
                if s % 200 == 0: print(f'step {s}/{len(batches)} loss {loss.item()*accum:.3f} {time.time()-t0:.0f}s')
            m.eval(); m.gradient_checkpointing_disable(); base.config.use_cache = True; m.save_pretrained(str(out_dir)); self.model, self.adapter = m, True
    return LM()


## 1. Settings and prompts
The same tickets, label lists and question wording as the Jev run, so instruments are compared on equal terms. The model must answer with one option from the list; answers are matched to the list, and anything unmatched counts as wrong.

In [ ]:
import json, time, gc, random, difflib, datetime, numpy as np, pandas as pd
from sklearn.metrics import f1_score
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar
JIRA = DATA_ROOT/'processed'/'jira'; TESTS = DATA_ROOT/'processed'/'jira_tests'; RESULTS = DATA_ROOT/'results'
RAW = RESULTS/'slm_raw'; RAW.mkdir(parents=True, exist_ok=True); ADAPTER_DIR = DATA_ROOT/'models'/'i6_lora'
SEED = 640; random.seed(SEED); FILES = sorted(JIRA.glob('*.parquet'))
LABELS = json.load(open(TESTS/'label_spaces.json'))
CLS = {'D1_issuetype': ('issuetype', 'What type of issue is this?'), 'D1_priority': ('priority', 'What priority should this issue be given?'),
       'D4_resolution': ('resolution', 'How is this issue most likely to be resolved?'), 'D5_component': ('proj_comp', 'Which project component (team) should handle this issue?')}
RET = {'D2_duplicate': 'Which of the earlier issues listed does this new issue duplicate?',
       'D6_cause': 'Which of the earlier issues listed caused the problem reported in this new issue?'}
def txt(x): return x if isinstance(x, str) else ''
def first_comp(s):
    try: c = json.loads(s)
    except Exception: return None
    return c[0] if c else None
def ticket(r, n=1200): return f"Project: {r['project']}\nSummary: {txt(r['summary'])}\nDescription: {txt(r['description'])[:n]}"
def cls_prompt(task, r, context=None):
    opts = '\n'.join(f'- {o}' for o in LABELS[task])
    ctx = ('\nSimilar earlier issues and how they were handled:\n' + context + '\n') if context else ''
    return (f"You are triaging an issue in a software issue tracker.\n\n{ticket(r)}\n{ctx}\nQuestion: {CLS[task][1]}\n"
            f"Options:\n{opts}\n\nAnswer with exactly one option from the list and nothing else.")
def ret_prompt(task, r, cands, texts):
    lines = '\n'.join(f"c{i+1}: {txt(texts.loc[c, 'summary'])} | {txt(texts.loc[c, 'description'])[:200]}" for i, c in enumerate(cands) if c in texts.index)
    return (f"New issue:\n{ticket(r)}\n\nEarlier issues:\n{lines}\nnone: None of the listed issues\n\nQuestion: {RET[task]}\n"
            "Answer with the identifier only (for example c3, or none).")
def to_label(text, space):
    t = text.strip().strip('.').strip('"').strip("'").splitlines()[0].strip() if text.strip() else ''
    low = {o.lower(): o for o in space}
    if t.lower() in low: return low[t.lower()]
    m = difflib.get_close_matches(t, space, n=1, cutoff=0.6)
    return m[0] if m else 'Other'
def wilson(k, n): lo, hi = proportion_confint(k, n, method='wilson'); return f'{100*lo:.1f}–{100*hi:.1f}'
def coverage(conf, ok, p):
    o = np.argsort(-conf); prec = np.cumsum(ok[o])/np.arange(1, len(o)+1); idx = np.where(prec >= p)[0]
    return round(100*(idx.max()+1)/len(o), 1) if len(idx) else 0.0
def ece(conf, ok, m=10):
    e = 0
    for a, b in zip(np.linspace(0, 1, m+1)[:-1], np.linspace(0, 1, m+1)[1:]):
        s = (conf > a) & (conf <= b)
        if s.any(): e += s.mean()*abs(ok[s].mean() - conf[s].mean())
    return round(e, 3)
tick = pd.read_parquet(TESTS/'jira_test_tickets.parquet').drop_duplicates('key').set_index('key', drop=False)
tests = {t: pd.read_csv(TESTS/f'testset_jira_{t}.csv') for t in list(CLS) + list(RET)}
HAS_CONTEXT = (TESTS/'context_i2_i3.csv').exists()
print(len(tick), 'test tickets; context from notebook 05 available:', HAS_CONTEXT)

## 2. Load the small language model
Qwen3-4B-Instruct-2507, an open 4-billion-parameter model under the Apache 2.0 licence (commercial use and fine-tuning permitted), runs on the Colab GPU with no API cost.

In [ ]:
t_gpu0 = time.time()
lm = load_model()

## 3. Run helper
Saves every answer to `results/slm_raw/*.jsonl` and records the GPU time per decision, which gives the cost per ticket. If the session stops, run all again: finished tickets are skipped and the fine-tuned adapter is reloaded.

In [ ]:
def run(name, items, max_new_tokens):
    """items: list of (key, prompt). Saves one JSON line per ticket; reruns skip finished tickets."""
    f = RAW/f'{name}.jsonl'; done = {}
    if f.exists():
        for l in open(f): o = json.loads(l); done[o['key']] = o
    todo = [(k, p) for k, p in items if k not in done]
    t0 = time.time()
    for b in range(0, len(todo), BATCH):
        chunk = todo[b:b+BATCH]
        outs, confs, ntok = lm.generate([p for _, p in chunk], max_new_tokens)
        with open(f, 'a') as fh:
            for (k, _), o, c, n in zip(chunk, outs, confs, ntok):
                rec = {'key': k, 'text': o, 'conf': c, 'prompt_tokens': n}; done[k] = rec; fh.write(json.dumps(rec) + '\n')
    secs = time.time() - t0
    print(f'{name}: {len(todo)} new, {len(done)} total, {secs:.0f}s')
    return done, secs
timing = {}

## 4. I5: the model as it is, with no training

In [ ]:
for t in CLS:
    items = [(k, cls_prompt(t, tick.loc[k])) for k in tests[t]['key']]
    _, timing[('I5', t)] = run(f'i5_{t}', items, 24)

I5 on duplicate (D2) and cause (D6) search: picks the matching issue from the I1 top-10 shortlist, as Jev does.

In [ ]:
need = set()
for t in RET: need |= {c for s in pd.read_csv(TESTS/f'i1_predictions_{t}.csv')['i1_top10'] for c in json.loads(s)}
texts = pd.concat([tick[['key', 'summary', 'description']]] + [pd.read_parquet(f, columns=['key', 'summary', 'description'], filters=[('key', 'in', list(need - set(tick.index)))]) for f in FILES]).drop_duplicates('key').set_index('key')
shortlist = {t: dict(zip(*pd.read_csv(TESTS/f'i1_predictions_{t}.csv')[['key', 'i1_top10']].values.T)) for t in RET}
for t in RET:
    items = [(k, ret_prompt(t, tick.loc[k], json.loads(shortlist[t][k]), texts)) for k in tests[t]['key'] if k in shortlist[t]]
    _, timing[('I5', t)] = run(f'i5_{t}', items, 6)

## 5. I6: fine-tuning data
Up to 3,000 examples per decision from the training period (before the validation and test periods), with at most 400 per class so rare classes are represented.

In [ ]:
created = pd.concat([pd.to_datetime(pd.read_parquet(f, columns=['created'])['created'], errors='coerce', utc=True) for f in FILES])
created = created[created >= pd.Timestamp('2000-01-01', tz='UTC')]; q70 = created.quantile(.7); del created; gc.collect()
pool = []
for f in FILES:
    d = pd.read_parquet(f, columns=['key', 'project', 'created', 'summary', 'description', 'issuetype', 'priority', 'resolution', 'components'])
    d['created'] = pd.to_datetime(d['created'], errors='coerce', utc=True)
    d = d[(d['created'] >= pd.Timestamp('2000-01-01', tz='UTC')) & (d['created'] < q70)]
    d['proj_comp'] = [f'{p} | {c}' if c else None for p, c in zip(d['project'], d['components'].map(first_comp))]
    pool.append(d.sample(min(len(d), 60000), random_state=SEED).drop(columns=['components', 'created'])); del d; gc.collect()
pool = pd.concat(pool, ignore_index=True)
train = []
for t, (col, _) in CLS.items():
    d = pool[pool[col].notna()].copy(); d['y'] = d[col].where(d[col].isin(LABELS[t]), 'Other')
    d = d.sample(frac=1, random_state=SEED); per_class = d[d.groupby('y').cumcount() < PER_CLASS_CAP]
    per_class = per_class.sample(min(len(per_class), N_TRAIN_PER_DECISION), random_state=SEED)
    train += [(cls_prompt(t, r), r['y']) for _, r in per_class.iterrows()]
random.shuffle(train); print(len(train), 'training examples (prompt, answer)')

## 6. I6: LoRA fine-tuning, then the same test sets
Low-rank adaptation (LoRA, rank 16) trains a small add-on to the model in about an hour. The adapter is saved to `Data/models/i6_lora`.

In [ ]:
t0 = time.time()
lm.train_lora(train, ADAPTER_DIR)
train_seconds = time.time() - t0; print(f'LoRA training {train_seconds/60:.1f} min')
for t in CLS:
    items = [(k, cls_prompt(t, tick.loc[k])) for k in tests[t]['key']]
    _, timing[('I6', t)] = run(f'i6_{t}', items, 24)

## 7. RQ3: the same models with company history in the prompt
D5 is run again with the five most similar earlier issues (from I2) and their components in the prompt. D2 and D6 are run again on the I1 + I2 hybrid shortlist. Skipped if notebook 05 has not run yet.

In [ ]:
if HAS_CONTEXT:
    ctx = pd.read_csv(TESTS/'context_i2_i3.csv')
    ctick = pd.read_parquet(TESTS/'context_tickets.parquet').drop_duplicates('key').set_index('key')
    ctick['proj_comp'] = [f'{p} | {c}' if c else None for p, c in zip(ctick['project'], ctick['components'].map(first_comp))]
    NICE = {'issuetype': 'type', 'priority': 'priority', 'resolution': 'resolution', 'proj_comp': 'component'}
    def context_text(t, k, n=5):
        row = ctx[(ctx['decision'] == t) & (ctx['key'] == k)]
        if row.empty: return None
        out = []
        for c in json.loads(row['i2_top10'].iloc[0])[:n]:
            if c in ctick.index:
                r = ctick.loc[c]; out.append(f"- {txt(r['summary'])[:150]} -> {NICE[CLS[t][0]]}: {r[CLS[t][0]]}")
        return '\n'.join(out)
    for tag in ('I5', 'I6'):
        if tag == 'I6': lm.use_adapter(True)
        else: lm.use_adapter(False)
        t = 'D5_component'
        items = [(k, cls_prompt(t, tick.loc[k], context_text(t, k))) for k in tests[t]['key']]
        _, timing[(tag + '+ctx', t)] = run(f'{tag.lower()}_ctx_{t}', items, 24)
    lm.use_adapter(False)
    hyb = {t: dict(zip(ctx[ctx['decision'] == t]['key'], ctx[ctx['decision'] == t]['hybrid_top10'])) for t in RET}
    extra = {c for t in RET for s in hyb[t].values() if isinstance(s, str) for c in json.loads(s)} - set(texts.index)
    texts = pd.concat([texts.reset_index()] + [pd.read_parquet(f, columns=['key', 'summary', 'description'], filters=[('key', 'in', list(extra))]) for f in FILES]).drop_duplicates('key').set_index('key')
    for t in RET:
        items = [(k, ret_prompt(t, tick.loc[k], json.loads(hyb[t][k]), texts)) for k in tests[t]['key'] if isinstance(hyb[t].get(k), str)]
        _, timing[('I5+ctx', t)] = run(f'i5_ctx_{t}', items, 6)
gpu_seconds = time.time() - t_gpu0

## 8. Results: classification

In [ ]:
def load(name):
    f = RAW/f'{name}.jsonl'
    return {json.loads(l)['key']: json.loads(l) for l in open(f)} if f.exists() else {}
i1 = {t: dict(zip(*pd.read_csv(TESTS/f'i1_predictions_{t}.csv')[['key', 'i1_pred']].values.T)) for t in CLS}
rows, preds = [], []
runs = [('I5', f'i5_{t}', t) for t in CLS] + [('I6', f'i6_{t}', t) for t in CLS] + [('I5 + I2 context', 'i5_ctx_D5_component', 'D5_component'), ('I6 + I2 context', 'i6_ctx_D5_component', 'D5_component')]
for inst, name, t in runs:
    a = load(name)
    if not a: continue
    ts = tests[t][tests[t]['key'].isin(a)]; y = ts['label'].values
    p = np.array([to_label(a[k]['text'], LABELS[t]) for k in ts['key']]); conf = np.array([a[k]['conf'] for k in ts['key']])
    ok = p == y; ok1 = np.array([i1[t].get(k) == lab for k, lab in zip(ts['key'], y)]); b, c = int((ok1 & ~ok).sum()), int((~ok1 & ok).sum())
    sec = timing.get((inst.split(' ')[0] + ('+ctx' if 'context' in inst else ''), t), np.nan)
    rows.append({'Decision': t, 'Instrument': f'{inst} ({MODEL_NAME})', 'Test rows': len(y), 'Accuracy %': round(100*ok.mean(), 1), '95% CI (Wilson)': wilson(int(ok.sum()), len(y)),
                 'Macro-F1': round(f1_score(y, p, average='macro', zero_division=0), 3), 'I1 accuracy %': round(100*ok1.mean(), 1),
                 'McNemar p (vs I1)': round(mcnemar([[0, b], [c, 0]], exact=True).pvalue, 4), 'Coverage % at 98% precision': coverage(conf, ok, .98),
                 'Coverage % at 90% precision': coverage(conf, ok, .90), 'ECE': ece(conf, ok), 'Unmatched answers %': round(100*np.mean([to_label(a[k]['text'], LABELS[t]) == 'Other' and a[k]['text'].strip().lower() != 'other' for k in ts['key']]), 1),
                 'GPU seconds per ticket': round(sec/len(y), 3) if sec == sec else None,
                 'Cost US$ per 1,000 tickets': round(1000*sec/len(y)/3600*UNITS_PER_HOUR*USD_PER_UNIT, 3) if sec == sec else None})
    preds.append(pd.DataFrame({'instrument': inst, 'decision': t, 'key': ts['key'], 'label': y, 'pred': p, 'conf': conf}))
slm = pd.DataFrame(rows); slm.to_csv(RESULTS/'rq1_jira_i5_i6.csv', index=False)
pd.concat(preds).to_csv(TESTS/'i5_i6_predictions_classification.csv', index=False); slm

## 9. Results: duplicate and cause search

In [ ]:
rrows = []
for inst, pre in (('I5 re-rank of I1 shortlist', 'i5_'), ('I5 re-rank of I1+I2 hybrid shortlist', 'i5_ctx_')):
    for t in RET:
        a = load(f'{pre}{t}')
        if not a: continue
        sl = shortlist[t] if pre == 'i5_' else hyb[t]
        hit = n = 0
        for k, ans in zip(tests[t]['key'], tests[t]['answers']):
            if k not in a or not isinstance(sl.get(k), str): continue
            n += 1; cands = json.loads(sl[k]); out = a[k]['text'].strip().lower()
            idx = int(''.join(ch for ch in out.split()[0] if ch.isdigit()) or 0) if out.startswith('c') else 0
            hit += bool(0 < idx <= len(cands) and cands[idx-1] in set(json.loads(ans)))
        rrows.append({'Decision': t, 'Instrument': f'{inst} ({MODEL_NAME})', 'Test rows': n, 'Recall@1 %': round(100*hit/n, 1), '95% CI': wilson(hit, n)})
rret = pd.DataFrame(rrows); rret.to_csv(RESULTS/'rq1_jira_i5_rerank.csv', index=False); rret

## 10. Cost and summary

In [ ]:
cost = pd.DataFrame([{'Item': 'All I5/I6 inference and LoRA training this session', 'GPU minutes': round(gpu_seconds/60, 1),
                      'LoRA training minutes': round(train_seconds/60, 1), 'Approx. compute units': round(gpu_seconds/3600*UNITS_PER_HOUR, 2),
                      'Approx. US$': round(gpu_seconds/3600*UNITS_PER_HOUR*USD_PER_UNIT, 2)}])
cost.to_csv(RESULTS/'rq1_i5_i6_cost.csv', index=False)
def mdt(d):
    try: return d.to_markdown(index=False)
    except Exception: return d.to_string()
lines = [f'# Step 06 summary - {datetime.datetime.now():%Y-%m-%d %H:%M}', f'Model: {MODEL_NAME}; LoRA r={LORA_R}', '',
         '## Classification', mdt(slm), '', '## Duplicate and cause re-ranking', mdt(rret), '', '## Cost', mdt(cost)]
(RESULTS/'step06_summary.md').write_text('\n'.join(lines)); print('\n'.join(lines))